## 1. Locate the repository and dataset

This cell finds the project root folder, forces Python to use the local `citylearn/` package from this repository, and defines the dataset paths for the TX climate case.

Do not change this cell unless your repository structure or dataset location is different.

In [1]:
import sys
from pathlib import Path

# 1) Find repo root = folder that contains "citylearn/"
HERE = Path.cwd()
REPO_DIR = None
for p in [HERE] + list(HERE.parents):
    if (p / "citylearn").exists():
        REPO_DIR = p
        break

if REPO_DIR is None:
    raise FileNotFoundError(
        "Couldn't find the repo root (a folder containing 'citylearn'). "
        "Make sure you cloned the repo and opened the notebook from somewhere inside it."
    )

# 2) Force local CityLearn (repo version)
sys.path.insert(0, str(REPO_DIR))
import citylearn
print("✅ citylearn loaded from:", citylearn.__file__)
print("✅ repo root:", REPO_DIR)

# 3) Dataset paths  – TX for the MPC comparison
CLIMATE = "TX"  # TX – cooling-dominated, June simulation window
DATASET_DIR = REPO_DIR / "data" / "datasets" / f"annex96_ce1_{CLIMATE.lower()}_neighborhood"
SCHEMA_PATH = DATASET_DIR / "schema.json"

print("\nClimate:", CLIMATE)
print("Dataset directory:", DATASET_DIR, " | exists:", DATASET_DIR.exists())
print("Schema path:", SCHEMA_PATH, " | exists:", SCHEMA_PATH.exists())

✅ citylearn loaded from: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\citylearn\__init__.py
✅ repo root: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1

Climate: TX
Dataset directory: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\data\datasets\annex96_ce1_tx_neighborhood  | exists: True
Schema path: c:\Users\yusuf\Documents\Python_Projects\CITYLEARN - Q4\annex96_common_exercise_1\data\datasets\annex96_ce1_tx_neighborhood\schema.json  | exists: True


## 2. Configure Weights & Biases logging

This section connects the notebook to the shared W&B project.

Important points:

- `WANDB_ENTITY` is the shared team workspace.
- `WANDB_PROJECT` is the shared W&B project where all controller results will appear.
- `log_to_wandb(...)` creates one W&B run per controller.
- For each controller, the function logs:
  - `district_load` as a time-series for load-tracking plots.
  - `NMBE [%]` as a scalar KPI for controller comparison.
  - `CV-RMSE [%]` as a scalar KPI for controller comparison.
  - `Temp Comfort violation [%]` as a scalar KPI for thermal-comfort comparison.
  - an interactive indoor-temperature plot when `building_temps` is provided.

Before running this notebook for the first time, log in from the terminal or notebook:

```
wandb login
```

Use your own W&B account. Do not share API keys.

In [2]:
import os
os.environ["WANDB_SILENT"] = "true"

import wandb
import plotly.graph_objects as go

WANDB_ENTITY="CityLearn-TeamB"
WANDB_PROJECT="CityLearn"

def log_to_wandb(controller_name, district_load, kpis=None, building_temps=None):
    
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        name=controller_name,
        reinit=True,
        settings=wandb.Settings(console="off")
    )

    # KPIs
    if kpis is not None:
        wandb.log({
            "NMBE [%]": float(kpis["NMBE [%]"]),
            "CV-RMSE [%]": float(kpis["CV-RMSE [%]"]),
            "Temp Comfort violation [%]": float(kpis["Temp Comfort violation [%]"]),
        })

    # Load time-series
    wandb.define_metric("hour")
    wandb.define_metric("district_load", step_metric="hour")
    for t in range(len(district_load)):
        wandb.log({
            "hour": t,
            "district_load": float(district_load[t]),
        })

    # Temperature logging
    if building_temps is not None:

        T = building_temps.shape[0]

        fig = go.Figure()

        # Comfort band 22–26 °C
        fig.add_hrect(
            y0=22,
            y1=26,
            fillcolor="green",
            opacity=0.12,
            line_width=0,
            annotation_text="Comfort band [22–26 °C]",
            annotation_position="top right"
        )

        fig.add_hline(y=22, line_dash="dash", line_color="red", line_width=1)
        fig.add_hline(y=26, line_dash="dash", line_color="red", line_width=1)

        # 25 buildings
        for b in range(building_temps.shape[1]):
            fig.add_trace(go.Scatter(
                x=list(range(T)),
                y=building_temps[:, b],
                mode="lines",
                line=dict(color="rgba(70,130,180,0.2)", width=1),
                showlegend=False
            ))

        # Mean line
        mean_temp = building_temps.mean(axis=1)
        fig.add_trace(go.Scatter(
            x=list(range(T)),
            y=mean_temp,
            mode="lines",
            line=dict(color="steelblue", width=3),
            name="Mean temperature"
        ))

        fig.update_layout(
            title=f"{controller_name} - Indoor Temperature",
            xaxis_title="Hour",
            yaxis_title="Temperature [°C]"
        )

        wandb.log({
            "temperature_plot": wandb.Plotly(fig)
        })

    wandb.finish()

## 3. Load the TX district target

The TX simulation window does not start at dataset timestep 0, it starts in the month June. For this assignment, the relevant window is:

- Start timestep: `3624`
- End timestep: `4343`
- Total duration: 720 hours, corresponding to the 30-day (1 month) simulation period.

The district target is sliced using the same timestep window so that:

```
district_target[0]
```

matches the first simulated hour of the controller output arrays.

In [3]:
import pandas as pd
import warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.INFO)

# TX simulation does NOT start at timestep 0 (Simulate in June).
TX_sim_start = 3624
TX_sim_end = 4343

# Slice the CSV so district_target[0] == first simulated hour.
# All episode arrays (net_electricity_consumption, building temps, etc.)
# use the same simulation-relative indexing: index 0 = first sim hour.
district_target_df = pd.read_csv(DATASET_DIR / "district_target.csv")
district_target = district_target_df["district_load_target"].values[TX_sim_start : TX_sim_end + 1]

## 4. Create the CityLearn environment

This notebook uses a decentralized CityLearn setup:

```
central_agent=False
```

This means each building has its own local action structure. The environment is reset before running each controller.

In [4]:
from citylearn.citylearn import CityLearnEnv

env = CityLearnEnv(
    schema=str(SCHEMA_PATH),
    root_directory=str(DATASET_DIR),
    central_agent=False,        
)

observations, info = env.reset()
print("Number of buildings:", len(env.buildings))

Number of buildings: 25


## 5. Run and evaluate `HourRBC`

This section runs the `HourRBC` controller.

After the simulation loop finishes, the district-level electricity consumption is stored as:

```
district_load_HourRBC = np.array(env.net_electricity_consumption)
```

The indoor temperature trajectories are then extracted from each building:

```
HourRBC_building_temps = np.array([
    b.energy_simulation.indoor_dry_bulb_temperature[:720]
    for b in env.buildings
]).T
```

This produces a `(720, 25)` array: 720 hourly values for 25 buildings.

The KPIs are computed using both the district load and building temperatures:

```
kpis_HourRBC = compute_kpis(
    district_target,
    district_load_HourRBC,
    HourRBC_building_temps
)
```

At this point, the controller results are ready to be uploaded to W&B.


In [5]:
import numpy as np
from citylearn.agents.rbc import HourRBC
from SERVER.KPIs import compute_kpis

obs, info = env.reset()
agent = HourRBC(env)

while not env.terminated:
    actions = agent.predict(obs)
    obs, reward, terminated, truncated, info = env.step(actions)

district_load_HourRBC = np.array(env.net_electricity_consumption)

# Extract building temperatures
HourRBC_building_temps = np.array([
    b.energy_simulation.indoor_dry_bulb_temperature[:720]
    for b in env.buildings
]).T  # shape: (720, 25)

# Evaluate KPI's
kpis_HourRBC = compute_kpis(district_target, district_load_HourRBC, HourRBC_building_temps)


### Upload `HourRBC` results to W&B

This cell sends the `HourRBC` results to the shared W&B project.

The run name will be `HourRBC`. In the W&B dashboard, this run will contain:

- the full `district_load` time-series,
- the `NMBE [%]` KPI,
- the `CV-RMSE [%]` KPI,
- the `Temp Comfort violation [%]` KPI,
- an interactive indoor-temperature plot showing all 25 buildings and the mean temperature.

In [6]:
# Log results to WandB server
log_to_wandb(
    controller_name="HourRBC_Decentralized",
    district_load=district_load_HourRBC,
    kpis=kpis_HourRBC,
    building_temps=HourRBC_building_temps
)

In [7]:
wandb.finish()